# AegisRAG — Merge Adapters & Export GGUF

Merges the SFT and DPO LoRA/DoRA adapters into the base Qwen2.5-7B-Instruct model,
converts to F16 GGUF, then quantizes to Q4_K_M (`aegis_final.gguf`).

**Run cells top to bottom. Each cell is independent and labelled.**

## Cell 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {vram:.1f} GB')

## Cell 2 — Install dependencies

In [ ]:
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 bitsandbytes==0.44.1
print('Dependencies installed.')

## Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## Cell 4 — Set paths

Edit `DRIVE_CHECKPOINTS` if your checkpoints folder is in a different Drive location.

In [ ]:
import os
from pathlib import Path

# ── Edit this if needed ───────────────────────────────────────────────
REPO_ROOT        = Path('/content/Demo')           # AegisRAG repo root on Colab
DRIVE_CHECKPOINTS = Path('/content/drive/MyDrive/checkpoints')  # your Drive folder
# ─────────────────────────────────────────────────────────────────────

SFT_ADAPTER  = REPO_ROOT / 'checkpoints' / 'generator_sft'
DPO_ADAPTER  = REPO_ROOT / 'checkpoints' / 'dpo'
MERGED_DIR   = REPO_ROOT / 'checkpoints' / 'generator_merged_new'
F16_GGUF     = REPO_ROOT / 'checkpoints' / 'generator.F16.gguf'
FINAL_GGUF   = REPO_ROOT / 'checkpoints' / 'aegis_final.gguf'
LLAMA_CPP    = Path('/content/Demo/llama.cpp')

print('Repo root :', REPO_ROOT)
print('Drive checkpoints:', DRIVE_CHECKPOINTS)
print('SFT adapter path :', SFT_ADAPTER)
print('DPO adapter path :', DPO_ADAPTER)

## Cell 5 — Copy adapters from Google Drive

In [ ]:
import shutil

ckpt_dir = REPO_ROOT / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

for name in ['generator_sft', 'dpo']:
    src = DRIVE_CHECKPOINTS / name
    dst = ckpt_dir / name
    if not src.exists():
        print(f'WARNING: {src} not found on Drive — skipping')
        continue
    if dst.exists():
        print(f'{dst} already exists — skipping copy')
    else:
        shutil.copytree(src, dst)
        print(f'Copied {src} -> {dst}')

print('\nCheckpoints folder:')
!ls -lh {ckpt_dir}

## Cell 6 — Clone llama.cpp (skip if already cloned)

In [ ]:
if not LLAMA_CPP.exists():
    !git clone https://github.com/ggerganov/llama.cpp.git {LLAMA_CPP}
    print('Cloned llama.cpp')
else:
    print('llama.cpp already present, pulling latest...')
    !git -C {LLAMA_CPP} pull

# Install Python deps for the convert script
!pip install -q -r {LLAMA_CPP}/requirements.txt
print('llama.cpp Python requirements installed.')

## Cell 7 — Build llama-quantize binary

In [ ]:
quantize_bin = LLAMA_CPP / 'llama-quantize'
if not quantize_bin.exists():
    print('Building llama-quantize...')
    !make -C {LLAMA_CPP} llama-quantize -j4
else:
    print('llama-quantize already built.')

!ls -lh {quantize_bin}

## Cell 8 — Add llama.cpp to PATH and verify

In [ ]:
os.environ['PATH'] = str(LLAMA_CPP) + ':' + os.environ['PATH']

# Make convert script executable
convert_script = LLAMA_CPP / 'convert_hf_to_gguf.py'
if convert_script.exists():
    os.chmod(convert_script, 0o755)
    print('convert_hf_to_gguf.py found and made executable.')
else:
    # Some versions use a different name
    alt = LLAMA_CPP / 'convert.py'
    if alt.exists():
        os.chmod(alt, 0o755)
        print(f'Using older convert.py: {alt}')
    else:
        print('ERROR: No convert script found in llama.cpp — check the clone.')

import shutil as _shutil
found = _shutil.which('convert_hf_to_gguf.py') or _shutil.which('convert.py')
print('convert script on PATH:', found)
print('llama-quantize on PATH:', _shutil.which('llama-quantize'))

## Cell 9 — Merge SFT adapter into base model

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT))

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'

print('Loading base model (this takes ~5 min)...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='cpu',          # CPU merge — safer than GPU for large models
    trust_remote_code=True,
)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print('Base model loaded.')

if SFT_ADAPTER.exists():
    print(f'Applying SFT adapter from {SFT_ADAPTER}...')
    model = PeftModel.from_pretrained(model, str(SFT_ADAPTER))
    model = model.merge_and_unload()
    print('SFT adapter merged.')
else:
    print(f'WARNING: SFT adapter not found at {SFT_ADAPTER} — skipping.')

## Cell 10 — Apply DPO adapter on top

In [ ]:
if DPO_ADAPTER.exists():
    print(f'Applying DPO adapter from {DPO_ADAPTER}...')
    model = PeftModel.from_pretrained(model, str(DPO_ADAPTER))
    model = model.merge_and_unload()
    print('DPO adapter merged.')
else:
    print(f'WARNING: DPO adapter not found at {DPO_ADAPTER} — skipping.')

## Cell 11 — Save merged model to disk

In [ ]:
MERGED_DIR.mkdir(parents=True, exist_ok=True)
print(f'Saving merged model to {MERGED_DIR}...')
model.save_pretrained(MERGED_DIR)
tok.save_pretrained(MERGED_DIR)
print('Saved.')
!ls -lh {MERGED_DIR}

# Free GPU/CPU memory before conversion
del model
import gc
gc.collect()
torch.cuda.empty_cache()
print('Memory freed.')

## Cell 12 — Convert merged model to F16 GGUF

In [ ]:
convert_script = str(LLAMA_CPP / 'convert_hf_to_gguf.py')
if not Path(convert_script).exists():
    convert_script = str(LLAMA_CPP / 'convert.py')   # older llama.cpp

print(f'Running: {convert_script}')
!python {convert_script} \
    {MERGED_DIR} \
    --outfile {F16_GGUF} \
    --outtype f16

print('\nF16 GGUF size:')
!ls -lh {F16_GGUF}

## Cell 13 — Quantize F16 → Q4_K_M (final model)

In [ ]:
quantize_bin = str(LLAMA_CPP / 'llama-quantize')
print(f'Quantizing {F16_GGUF} -> {FINAL_GGUF} (Q4_K_M)...')
!{quantize_bin} {F16_GGUF} {FINAL_GGUF} Q4_K_M

print('\nFinal GGUF size:')
!ls -lh {FINAL_GGUF}

## Cell 14 — Copy final GGUF back to Google Drive

In [ ]:
drive_out = DRIVE_CHECKPOINTS / 'aegis_final.gguf'
DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f'Copying {FINAL_GGUF} -> {drive_out}...')
shutil.copy2(FINAL_GGUF, drive_out)
print('Saved to Drive.')
!ls -lh {drive_out}

## Cell 15 — (Optional) Download directly to your computer

In [ ]:
# Uncomment to trigger a browser download (~4.4 GB)
# from google.colab import files
# files.download(str(FINAL_GGUF))

## Cell 16 — Cleanup intermediate files (optional, saves disk space)

In [ ]:
# Uncomment to delete intermediate files and free ~50 GB
# shutil.rmtree(MERGED_DIR, ignore_errors=True)
# F16_GGUF.unlink(missing_ok=True)
# print('Cleaned up intermediate files.')
print('Skipping cleanup — uncomment above lines if you want to free disk space.')